# Setup y Lectura de Datos

In [ ]:
%pip install causal-learn -q

In [ ]:
import pyspark.sql.functions as sf
from pyspark.storagelevel import StorageLevel
import pandas as pd
import numpy as np
from causallearn.utils.PCUtils.BackgroundKnowledge import BackgroundKnowledge
from causallearn.graph.GraphNode import GraphNode
import time
import io
from contextlib import redirect_stdout
from causallearn.search.ConstraintBased.FCI import fci
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Patch
from matplotlib.lines import Line2D

BU = 'MEX'
CONFUSORES = ['territorio', 'canal', 'subcanal', 'tamano_cliente']

CAPACIDADES_CONTINUAS = ['Digital', 'Multicategory', 'Coolers', 'PedidoSugerido']
CAPACIDADES_BINARIAS  = ['RTM', 'POS'] 
CAPACIDADES_TODAS = CAPACIDADES_CONTINUAS + CAPACIDADES_BINARIAS
CAPACIDADES_LOG = ['Coolers']

OUTCOME_FUENTE = 'ingreso_neto_core_real'
OUTCOME = 'log_ingreso_neto_core'


cols_pedidas = ['id_cliente'] + CONFUSORES + CAPACIDADES_TODAS + [OUTCOME_FUENTE]
schema_cols = spark.read.parquet(ruta_panel).columns
faltantes = [c for c in cols_pedidas if c not in schema_cols]
if faltantes:
    raise ValueError(f"Faltan columnas: {faltantes}")

panel = spark.read.parquet(ruta_panel).select(*cols_pedidas).persist(StorageLevel.MEMORY_AND_DISK)

print(f"Panel: {panel.count():,} filas × {panel.select('id_cliente').distinct().count():,} PDVs")

exprs = []
for cap in CAPACIDADES_TODAS:
    exprs.append(sf.avg(sf.col(cap)).alias(cap))
exprs.append(sf.avg(sf.log(sf.col(OUTCOME_FUENTE) + 1)).alias(OUTCOME))
for conf in CONFUSORES:
    exprs.append(sf.first(sf.col(conf), ignorenulls=True).alias(conf))

matriz_pdv = panel.groupBy('id_cliente').agg(*exprs).toPandas()

matriz_proc = matriz_pdv.copy()
for col in CAPACIDADES_LOG:
    matriz_proc[col] = np.log1p(matriz_proc[col])

cols_a_std = CAPACIDADES_TODAS + [OUTCOME]
matriz_std = matriz_proc.copy()
for col in cols_a_std:
    mu, sigma = matriz_std[col].mean(), matriz_std[col].std()
    if sigma > 0:
        matriz_std[col] = (matriz_std[col] - mu) / sigma

print(f"\nMatriz lista: {len(matriz_std):,} PDVs × {len(cols_a_std)} numéricas + {len(CONFUSORES)} categóricas")
print(f"Capacidades: {CAPACIDADES_TODAS}")
print(f"Confusores: {CONFUSORES}")
panel.unpersist()

# Cardinalidad de confusores

In [ ]:
print("Cardinalidad y distribución de confusores:\n")
for conf in CONFUSORES:
    n_unique = matriz_pdv[conf].nunique()
    print(f"  {conf:>16}: {n_unique} niveles únicos")
    if n_unique <= 15:
        vc = matriz_pdv[conf].value_counts(dropna=False)
        for cat, n in vc.head(15).items():
            pct = n / len(matriz_pdv) * 100
            print(f"      {str(cat):<25} {n:>10,}  ({pct:.1f}%)")
    else:
        # solo top 5 si son muchos
        vc = matriz_pdv[conf].value_counts(dropna=False).head(10)
        for cat, n in vc.items():
            pct = n / len(matriz_pdv) * 100
            print(f"      {str(cat):<25} {n:>10,}  ({pct:.1f}%)")
        print(f"      y ({n_unique - 5} niveles más)")
    print()

# Encoding Cofusores

In [ ]:
TOP_N_CANAL = 5
TOP_N_TERRITORIO = 5
TOP_N_SUBCANAL = 10

MAPEO_TAMANO = {
    'MICRO': 1, 'CHICO': 1, 'Sin Asignar': 1,
    'MEDIANO': 2,
    'GRANDE': 3, 'EXT-GDE': 3,
}
LABEL_TAMANO = {1: 'PEQUEÑO', 2: 'MEDIANO', 3: 'GRANDE'}

matriz_enc = matriz_std.copy()

matriz_enc['Tamaño'] = matriz_enc['tamano_cliente'].map(MAPEO_TAMANO)
if matriz_enc['Tamaño'].isnull().any():
    raise ValueError("Sin mapear en tamano_cliente")

print("Tamaño:")
for k in [1, 2, 3]:
    n = (matriz_enc['Tamaño'] == k).sum()
    print(f"  {LABEL_TAMANO[k]}: {n:,} ({n/len(matriz_enc)*100:.1f}%)")

def encode_top_n(serie, top_n, prefix):
    if top_n >= serie.nunique():
        grupo = serie.astype(str)
    else:
        top = serie.value_counts().head(top_n).index.tolist()
        grupo = np.where(serie.isin(top), serie.astype(str), 'Otros')
    dummies = pd.get_dummies(grupo, prefix=prefix, drop_first=True).astype(int)
    return dummies, pd.Series(grupo).value_counts()

print(f"\nTerritorio (top {TOP_N_TERRITORIO} + Otros):")
dummies_terr, dist_terr = encode_top_n(matriz_enc['territorio'], TOP_N_TERRITORIO, 'terr')
print(f"  Niveles activos: {len(dist_terr)}, dummies: {len(dummies_terr.columns)}")

print(f"\nCanal (top {TOP_N_CANAL} + Otros):")
dummies_canal, dist_canal = encode_top_n(matriz_enc['canal'], TOP_N_CANAL, 'canal')
print(f"  Niveles activos: {len(dist_canal)}, dummies: {len(dummies_canal.columns)}")

print(f"\nSubcanal (top {TOP_N_SUBCANAL} + Otros):")
dummies_subcanal, dist_subcanal = encode_top_n(matriz_enc['subcanal'], TOP_N_SUBCANAL, 'subcanal')
print(f"  Niveles activos: {len(dist_subcanal)}, dummies: {len(dummies_subcanal.columns)}")

cols_caps = CAPACIDADES_TODAS
cols_out = [OUTCOME]
cols_tamano = ['Tamaño']
cols_terr = dummies_terr.columns.tolist()
cols_canal = dummies_canal.columns.tolist()
cols_subcanal = dummies_subcanal.columns.tolist()

matriz_discovery = pd.concat([
    matriz_enc[['id_cliente'] + cols_caps + cols_out + cols_tamano].reset_index(drop=True),
    dummies_terr.reset_index(drop=True),
    dummies_canal.reset_index(drop=True),
    dummies_subcanal.reset_index(drop=True),
], axis=1)

NODOS_GRAFO = cols_caps + cols_out + cols_tamano + cols_terr + cols_canal + cols_subcanal
NODOS_CONFUSORES = cols_tamano + cols_terr + cols_canal + cols_subcanal
NODO_OUTCOME = OUTCOME

print(f"\nMatriz lista — {BU}")
print(f"PDVs: {len(matriz_discovery):,}")
print(f"Total nodos del grafo: {len(NODOS_GRAFO)}")
print(f"  Capacidades ({len(cols_caps)}): {cols_caps}")
print(f"  Outcome (1): {cols_out}")
print(f"  Tamaño ordinal (1): {cols_tamano}")
print(f"  Territorio ({len(cols_terr)}): top {TOP_N_TERRITORIO} + Otros")
print(f"  Canal ({len(cols_canal)}): top {TOP_N_CANAL} + Otros")
print(f"  Subcanal ({len(cols_subcanal)}): top {TOP_N_SUBCANAL} + Otros")

# Background Knowledge para FCI

In [ ]:
nodes = [GraphNode(f'X{i+1}') for i in range(len(NODOS_GRAFO))]
node_map = {NODOS_GRAFO[i]: nodes[i] for i in range(len(NODOS_GRAFO))}

NODOS_CAPACIDADES = CAPACIDADES_TODAS
NODO_OUTCOME = OUTCOME

print("Nodos del grafo:")
for i, name in enumerate(NODOS_GRAFO):
    if name in NODOS_CAPACIDADES:
        rol = 'CAP'
    elif name == NODO_OUTCOME:
        rol = 'OUT'
    else:
        rol = 'CONF'
    print(f"  X{i+1:2d} = {name:<28} [{rol}]")

bk = BackgroundKnowledge()
n_forbidden = 0

# Se egenran 2 reglas claras como Bakckground Knoledege

#Regla 1: nada apunta a un confusor (son exógenos al modelo)
for destino in NODOS_CONFUSORES:
    for origen in NODOS_GRAFO:
        if origen != destino:
            bk.add_forbidden_by_node(node_map[origen], node_map[destino])
            n_forbidden += 1

# Regla 2: outcome no es padre de nada
for destino in NODOS_GRAFO:
    if destino != NODO_OUTCOME:
        bk.add_forbidden_by_node(node_map[NODO_OUTCOME], node_map[destino])
        n_forbidden += 1

print(f"Aristas prohibidas: {n_forbidden}")

# Discovery FCI+BK

In [ ]:
ALPHA_FULL = 1e-5

def discover_one_run(data, alpha=ALPHA_FULL):
    if isinstance(data, pd.DataFrame):
        X = data[NODOS_GRAFO].values.astype(float)
    else:
        X = data.astype(float)
    
    with redirect_stdout(io.StringIO()):
        g, _ = fci(
            dataset=X,
            independence_test_method='fisherz',
            alpha=alpha,
            background_knowledge=bk,
            verbose=False,
            show_progress=False,
        )
    
    adj = g.graph
    n = len(NODOS_GRAFO)
    directed, bidirected, o_directed, oo = [], [], [], []
    
    for i in range(n):
        for j in range(i+1, n):
            mi, mj = adj[i, j], adj[j, i]
            if mi == 0 and mj == 0:
                continue
            elif mi == -1 and mj == 1:
                directed.append((NODOS_GRAFO[i], NODOS_GRAFO[j]))
            elif mi == 1 and mj == -1:
                directed.append((NODOS_GRAFO[j], NODOS_GRAFO[i]))
            elif mi == 1 and mj == 1:
                a, b = sorted([NODOS_GRAFO[i], NODOS_GRAFO[j]])
                bidirected.append((a, b))
            elif mi == 2 and mj == 1:
                o_directed.append((NODOS_GRAFO[i], NODOS_GRAFO[j]))
            elif mi == 1 and mj == 2:
                o_directed.append((NODOS_GRAFO[j], NODOS_GRAFO[i]))
            elif mi == 2 and mj == 2:
                a, b = sorted([NODOS_GRAFO[i], NODOS_GRAFO[j]])
                oo.append((a, b))
            elif mi == -1 and mj == -1:
                a, b = sorted([NODOS_GRAFO[i], NODOS_GRAFO[j]])
                oo.append((a, b))
    
    return {'directed': directed, 'bidirected': bidirected, 'o_directed': o_directed, 'oo': oo}


print(f"Discovery sobre panel completo: N={len(matriz_discovery):,}, alpha={ALPHA_FULL}")

t0 = time.time()
result_full = discover_one_run(matriz_discovery, alpha=ALPHA_FULL)
elapsed = time.time() - t0

total = sum(len(v) for v in result_full.values())
print(f"✓ Completado en {elapsed:.0f}s")
print(f"\nAristas detectadas (total={total}):")
print(f"  Dirigidas (→):     {len(result_full['directed'])}")
print(f"  Bidirigidas (↔):   {len(result_full['bidirected'])}")
print(f"  Parciales (o→):    {len(result_full['o_directed'])}")
print(f"  Indeterminadas:    {len(result_full['oo'])}")

print(f"\nDirigidas:")
for src, dst in result_full['directed']:
    print(f"  {src:>22} → {dst}")

print(f"\nBidirigidas (confounding latente):")
for a, b in result_full['bidirected']:
    print(f"  {a:>22} ↔ {b}")

print(f"\nParciales (o→):")
for src, dst in result_full['o_directed']:
    print(f"  {src:>22} o→ {dst}")

print(f"\nIndeterminadas:")
for a, b in result_full['oo']:
    print(f"  {a:>22} — {b}")

# Verificación BK
violaciones = []
for src, dst in result_full['directed']:
    if dst in NODOS_CONFUSORES:
        violaciones.append(f"{src} → {dst}")
    if src == NODO_OUTCOME:
        violaciones.append(f"{src} → {dst}")

print(f"\nBackground Knoledge:")
print("Ok" if not violaciones else f"⚠ Existen violaciones: {violaciones}")

In [ ]:
#Visualización del DAG descubierto sobre panel completo

COLOR_CONF = '#888780'
COLOR_CAP  = '#D4537E'
COLOR_OUT  = '#1D9E75'

def colapsar(name):
    if name.startswith('terr_'):     return 'Territorio'
    if name.startswith('canal_'):    return 'Canal'
    if name.startswith('subcanal_'): return 'Subcanal'
    return name

aristas_v = []
for src, dst in result_full['directed']:
    src_c, dst_c = colapsar(src), colapsar(dst)
    if src_c != dst_c:
        aristas_v.append({'src': src_c, 'dst': dst_c, 'tipo': 'directed'})

for a, b in result_full['bidirected']:
    a_c, b_c = colapsar(a), colapsar(b)
    if a_c != b_c:
        aristas_v.append({'src': a_c, 'dst': b_c, 'tipo': 'bidirected'})

for src, dst in result_full['o_directed']:
    src_c, dst_c = colapsar(src), colapsar(dst)
    if src_c != dst_c:
        aristas_v.append({'src': src_c, 'dst': dst_c, 'tipo': 'partial'})

for a, b in result_full['oo']:
    a_c, b_c = colapsar(a), colapsar(b)
    if a_c != b_c:
        aristas_v.append({'src': a_c, 'dst': b_c, 'tipo': 'unknown'})

df_v = pd.DataFrame(aristas_v).drop_duplicates(subset=['src', 'dst', 'tipo']).reset_index(drop=True)

NODOS_CONF_VIS = ['Territorio', 'Canal', 'Subcanal', 'Tamaño']
NODOS_CAP_VIS = CAPACIDADES_TODAS
NODO_OUT_VIS = OUTCOME

pos = {}
for i, x in enumerate(NODOS_CONF_VIS):
    pos[x] = (0, 5.0 - i * (5.0 / max(len(NODOS_CONF_VIS) - 1, 1)))
for i, w in enumerate(NODOS_CAP_VIS):
    pos[w] = (5, 5.5 - i * (5.5 / max(len(NODOS_CAP_VIS) - 1, 1)))
pos[NODO_OUT_VIS] = (10, 2.75)

NODE_W, NODE_H = 1.6, 0.55

def rol_color(name):
    if name in NODOS_CONF_VIS: return COLOR_CONF
    if name == NODO_OUT_VIS:   return COLOR_OUT
    return COLOR_CAP

def label_corto(name):
    return 'Revenue' if name == OUTCOME else name

def dibujar_nodo(ax, name):
    x, y = pos[name]
    box = FancyBboxPatch(
        (x - NODE_W/2, y - NODE_H/2), NODE_W, NODE_H,
        boxstyle="round,pad=0.02,rounding_size=0.22",
        linewidth=1.5, edgecolor='black', facecolor=rol_color(name), zorder=3
    )
    ax.add_patch(box)
    ax.text(x, y, label_corto(name), ha='center', va='center',fontsize=9, fontweight='bold', color='white', zorder=4)

def punto_borde(p_o, p_d):
    dx, dy = p_d[0] - p_o[0], p_d[1] - p_o[1]
    if dx == 0 and dy == 0: return p_o
    a, b = NODE_W/2, NODE_H/2
    t = 1.0 / np.sqrt((dx/a)**2 + (dy/b)**2)
    return (p_o[0] + dx*t, p_o[1] + dy*t)

def dibujar_arista(ax, src, dst, tipo, rad=0.0):
    p1, p2 = pos[src], pos[dst]
    p1b = punto_borde(p1, p2)
    p2b = punto_borde(p2, p1)
    
    if tipo == 'directed':
        ax.annotate('', xy=p2b, xytext=p1b,arrowprops=dict(arrowstyle='->', color='#185FA5', lw=2.0,connectionstyle=f'arc3,rad={rad}'), zorder=2)
    elif tipo == 'bidirected':
        ax.annotate('', xy=p2b, xytext=p1b,arrowprops=dict(arrowstyle='<->', color='#E24B4A', lw=2.5,connectionstyle=f'arc3,rad={max(rad, 0.2)}'), zorder=2)
    elif tipo == 'partial':
        ax.annotate('', xy=p2b, xytext=p1b,arrowprops=dict(arrowstyle='->', color='#7A5BA8', lw=1.8,connectionstyle=f'arc3,rad={rad}'), zorder=2)
        ax.plot(*p1b, marker='o', markerfacecolor='white',markeredgecolor='#7A5BA8', markersize=8, zorder=3)
    elif tipo == 'unknown':
        ax.plot([p1b[0], p2b[0]], [p1b[1], p2b[1]],color='#999999', linewidth=1.8, zorder=2)

fig, ax = plt.subplots(figsize=(15, 11))

for n in pos.keys():
    dibujar_nodo(ax, n)

for _, r in df_v.iterrows():
    src_es_cap = r['src'] in NODOS_CAP_VIS
    dst_es_cap = r['dst'] in NODOS_CAP_VIS
    rad = 0.25 if (src_es_cap and dst_es_cap) else 0.0
    dibujar_arista(ax, r['src'], r['dst'], r['tipo'], rad=rad)

ax.set_xlim(-1.8, 11.8)
ax.set_ylim(-0.8, 6.2)
ax.set_aspect('equal')
ax.axis('off')

legend_elements = [
    Patch(facecolor=COLOR_CONF, label='Confusor (X)'),
    Patch(facecolor=COLOR_CAP, label='Capacidad (W)'),
    Patch(facecolor=COLOR_OUT, label='Outcome (Y)'),
    Line2D([0], [0], color='#185FA5', lw=2.0, label='→ dirigida'),
    Line2D([0], [0], color='#E24B4A', lw=2.5, label='↔ confounding latente'),
    Line2D([0], [0], color='#7A5BA8', lw=1.8, label='o→ parcialmente orientada'),
]
ax.legend(handles=legend_elements, loc='lower center',
          ncol=3, bbox_to_anchor=(0.5, -0.04), fontsize=10, framealpha=0.95)

ax.set_title(
    f'DAG descubierto por FCI+BK sobre panel completo de {BU}\n'
    f'N={len(matriz_discovery):,} PDVs, alpha={ALPHA_FULL}',
    fontsize=13, fontweight='bold', pad=10
)

plt.tight_layout()
plt.show()

# DAG Final (Validación Manual y de Negocio)

In [ ]:

# Aristas del DAG 
DAG_POR_BU = {
    'MEX': [
        'Digital → Revenue',
        'Coolers → Revenue',
        'PedidoSugerido → Revenue',

        'Digital → Coolers',
        'Digital ↔ Multicategory',
        'Digital ↔ PedidoSugerido',
        'Digital ↔ POS',
        'Multicategory ↔ PedidoSugerido',
        'Multicategory ↔ RTM',
        'Multicategory ↔ POS',
        'POS ↔ PedidoSugerido',

        'Territorio → Digital',
        'Subcanal ↔ Digital',
        'Territorio → Multicategory',
        'Subcanal ↔ Multicategory',
        'Territorio → Coolers',
        'Canal → Coolers',
        'Subcanal ↔ Coolers',
        'Tamaño → PedidoSugerido',
        'Territorio → PedidoSugerido',
        'Canal → PedidoSugerido',
        'Subcanal ↔ PedidoSugerido',
        'Territorio → RTM',
        'Subcanal ↔ RTM',
        'Tamaño → POS',
        'Territorio → POS',

        'Tamaño → Revenue',
        'Territorio → Revenue',
        'Subcanal → Revenue',
    ]
}

In [ ]:
if BU not in DAG_POR_BU:
    raise ValueError(f"No hay DAG definido para BU={BU}. "
                     f"BUs disponibles: {list(DAG_POR_BU.keys())}")

ARISTAS_DAG = DAG_POR_BU[BU]

if not ARISTAS_DAG:
    raise ValueError(f"DAG para BU={BU} está vacío. "
                     f"Completá DAG_POR_BU['{BU}'] antes de continuar.")

print(f"DAG cargado para BU={BU}: {len(ARISTAS_DAG)} aristas")

def parse_arista(desc):
    for sep, tipo in [(' ↔ ', 'bidirected'), (' → ', 'directed')]:
        if sep in desc:
            a, b = desc.split(sep)
            return a, b, tipo
    return None, None, None

aristas_dir, aristas_bidir = [], []
for desc in ARISTAS_DAG:
    a, b, tipo = parse_arista(desc)
    if a is None:
        print(f"⚠ No parseable: {desc}")
        continue
    if tipo == 'directed':
        aristas_dir.append((a, b))
    elif tipo == 'bidirected':
        aristas_bidir.append((a, b))

print(f"DAG validado — {BU}")
print(f"  Dirigidas:    {len(aristas_dir)}")
print(f"  Bidirigidas:  {len(aristas_bidir)}")
print(f"  Total:        {len(aristas_dir) + len(aristas_bidir)}")

In [ ]:
COLOR_CONF, COLOR_CAP, COLOR_OUT = '#888780', '#D4537E', '#1D9E75'

NODOS_CONF_VIS = ['Territorio', 'Canal', 'Subcanal', 'Tamaño']
NODOS_CAP_VIS  = CAPACIDADES_TODAS
NODO_OUT_VIS   = 'Revenue'

pos = {}
for i, x in enumerate(NODOS_CONF_VIS):
    pos[x] = (0, 5.5 - i * (5.5 / max(len(NODOS_CONF_VIS) - 1, 1)))
for i, w in enumerate(NODOS_CAP_VIS):
    pos[w] = (5, 5.5 - i * (5.5 / max(len(NODOS_CAP_VIS) - 1, 1)))
pos[NODO_OUT_VIS] = (10, 2.75)

NODE_W, NODE_H = 1.6, 0.55

def rol_color(name):
    if name in NODOS_CONF_VIS: return COLOR_CONF
    if name == NODO_OUT_VIS:   return COLOR_OUT
    return COLOR_CAP

def dibujar_nodo(ax, name):
    x, y = pos[name]
    box = FancyBboxPatch(
        (x - NODE_W/2, y - NODE_H/2), NODE_W, NODE_H,
        boxstyle="round,pad=0.02,rounding_size=0.22",
        linewidth=1.5, edgecolor='black', facecolor=rol_color(name), zorder=3)
    ax.add_patch(box)
    ax.text(x, y, name, ha='center', va='center',fontsize=9, fontweight='bold', color='white', zorder=4)

def punto_borde(p_o, p_d):
    dx, dy = p_d[0] - p_o[0], p_d[1] - p_o[1]
    if dx == 0 and dy == 0: return p_o
    a, b = NODE_W/2, NODE_H/2
    t = 1.0 / np.sqrt((dx/a)**2 + (dy/b)**2)
    return (p_o[0] + dx*t, p_o[1] + dy*t)

def dibujar_arista(ax, src, dst, tipo, rad=0.0):
    if src not in pos or dst not in pos:
        print(f"⚠ Nodo no posicionado: {src} o {dst}")
        return
    p1, p2 = pos[src], pos[dst]
    p1b, p2b = punto_borde(p1, p2), punto_borde(p2, p1)
    if tipo == 'directed':
        ax.annotate('', xy=p2b, xytext=p1b,arrowprops=dict(arrowstyle='->', color='#185FA5', lw=2.0,connectionstyle=f'arc3,rad={rad}'), zorder=2)
    elif tipo == 'bidirected':
        ax.annotate('', xy=p2b, xytext=p1b,arrowprops=dict(arrowstyle='<->', color='#E24B4A', lw=2.5,connectionstyle=f'arc3,rad={max(rad, 0.2)}'), zorder=2)

fig, ax = plt.subplots(figsize=(15, 11))

for n in pos.keys():
    dibujar_nodo(ax, n)

for src, dst in aristas_dir:
    src_es_cap = src in NODOS_CAP_VIS
    dst_es_cap = dst in NODOS_CAP_VIS
    rad = 0.25 if (src_es_cap and dst_es_cap) else 0.0
    dibujar_arista(ax, src, dst, 'directed', rad=rad)

for src, dst in aristas_bidir:
    dibujar_arista(ax, src, dst, 'bidirected', rad=0.25)

ax.set_xlim(-1.8, 11.8); ax.set_ylim(-0.8, 6.2)
ax.set_aspect('equal'); ax.axis('off')

legend_elements = [
    Patch(facecolor=COLOR_CONF, label='Confusor (X)'),
    Patch(facecolor=COLOR_CAP, label='Capacidad (W)'),
    Patch(facecolor=COLOR_OUT, label='Outcome (Y)'),
    Line2D([0], [0], color='#185FA5', lw=2.0, label='→ dirigida'),
    Line2D([0], [0], color='#E24B4A', lw=2.5, label='↔ confounding latente'),
]
ax.legend(handles=legend_elements, loc='lower center',ncol=3, bbox_to_anchor=(0.5, -0.04), fontsize=10, framealpha=0.95)

ax.set_title(
    f'DAG validado por dominio — {BU}\n'
    f'Dirigidas: {len(aristas_dir)} | Bidirigidas: {len(aristas_bidir)}',
    fontsize=13, fontweight='bold', pad=10)

plt.tight_layout()
plt.show()